# 01 – ColoredMNIST with Spurious Correlations (ERM Baseline)

## 0) Setup and Imports


In [18]:
import os, math, random, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split

import torchvision
from torchvision import transforms
from torchvision.datasets import MNIST

import matplotlib.pyplot as plt


np.random.seed(0); random.seed(0); torch.manual_seed(0)



## 1) ColoredMNIST Dataset (spurious feature = **color**)

- Binary classification: digits in {0..4} → label y=1, digits in {5..9} → label y=0.
- Digit color z: {red, green}. During training, color is correlated
  with the label with probability ρ (0.95 or 0.99).
- Groups defined by (y, z) → 4 groups. **Minority groups** are those where color and label
  are misaligned.
- `val_clean`: group-balanced validation set (~200 samples per group).
- **Test splits:**
  - `test_spurious`: same color-label distribution as training.
  - `test_clean`: color is random, spurious cue is unrelated.


In [ ]:
class ColoredMNIST(Dataset):
    def __init__(self, root, split="train", rho=0.95, transform=None, download=True):
        assert split in ["train", "val_clean", "test_spurious", "test_clean"]
        self.split, self.rho = split, rho
        self.transform = transform
        
        # Load base MNIST (grayscale 28x28 images)
        train_flag = (split in ["train", "val_clean"])
        self.base = MNIST(root=root, train=train_flag, download=download)
        self.X = self.base.data.float() / 255.0  # shape (N, 28, 28), normalized values in [0, 1]
        self.y_digit = self.base.targets  # original digit labels 0..9
        
        # Binary label: y=1 if digit in {0,1,2,3,4}, y=0 if digit in {5,6,7,8,9}
        self.y = (self.y_digit < 5).long()
        
        # Assign spurious attribute z based on split type and correlation strength rho
        rng = np.random.RandomState(42 if split != "train" else 0)
        y_np = self.y.numpy()
        
        if split == "test_clean":
            z = rng.randint(0, 2, size=len(y_np)) # We assign colors randomly 
        else:
            # z matches y with probability rho (spurious correlation)
            match = rng.rand(len(y_np)) < rho
            z = np.where(match, y_np, 1 - y_np)
        
        self.z = torch.tensor(z).long()
        
        # Encode group membership: g = y * 2 + z  → {0, 1, 2, 3}
        self.groups = self.y * 2 + self.z
        
        if split == "val_clean":
            # Build a group-balanced subset: take at most 200 samples per group
            ids_per_group = [torch.where(self.groups == g)[0].tolist() for g in range(4)]
            min_n = min(len(ids) for ids in ids_per_group)
            min_n = min(min_n, 200)  # cap at 200 per group
            balanced_ids = []
            for ids in ids_per_group:
                rng.shuffle(ids)
                balanced_ids.extend(ids[:min_n])
            balanced_ids = sorted(balanced_ids)
            
            self.X = self.X[balanced_ids]
            self.y = self.y[balanced_ids]
            self.z = self.z[balanced_ids]
            self.groups = self.groups[balanced_ids]
        
    def __len__(self): return len(self.y)
    
    def colorize(self, img, z):
        # Convert grayscale image (28x28) to a 3-channel RGB tensor with a color tint.
        # z=0 → red tint, z=1 → green tint (channel scaling)
        img3 = torch.stack([img, img, img], dim=0)  # shape (3, 28, 28)
        if z == 0:
            scale = torch.tensor([1.0, 0.5, 0.5]).view(3, 1, 1)  # red tint
        else:
            scale = torch.tensor([0.5, 1.0, 0.5]).view(3, 1, 1)  # green tint
        return (img3 * scale).clamp(0, 1)
    
    def __getitem__(self, idx):
        x = self.X[idx]; y = int(self.y[idx]); z = int(self.z[idx]); g = int(self.groups[idx])
        x = self.colorize(x, z)
        if self.transform is not None:
            x = self.transform(x)
        return x, y, z, g


### Prepare dataset splits


In [20]:
def make_splits(root="./data", rho=0.95, batch_size=128):
    tfm = transforms.Compose([])  # tensors are already prepared in the Dataset
    train_ds = ColoredMNIST(root, split="train",        rho=rho, transform=tfm, download=True)
    val_clean = ColoredMNIST(root, split="val_clean",   rho=rho, transform=tfm, download=True)
    test_sp   = ColoredMNIST(root, split="test_spurious", rho=rho, transform=tfm, download=True)
    test_clean = ColoredMNIST(root, split="test_clean",  rho=rho, transform=tfm, download=True)
    
    loaders = {
        "train":         DataLoader(train_ds,  batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True),
        "val_clean":     DataLoader(val_clean, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True),
        "test_spurious": DataLoader(test_sp,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True),
        "test_clean":    DataLoader(test_clean,batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True),
    }
    return train_ds, val_clean, test_sp, test_clean, loaders


### Visualize examples (one row per group)


In [21]:
def show_examples(ds, n_per_group=6):
    # Display n_per_group examples for each of the 4 groups in a grid
    fig, axes = plt.subplots(4, n_per_group, figsize=(n_per_group * 2, 8))
    counts = [0, 0, 0, 0]
    for i in range(len(ds)):
        x, y, z, g = ds[i]
        if counts[g] >= n_per_group:
            continue
        ax = axes[g, counts[g]]
        ax.imshow(x.permute(1, 2, 0).numpy())
        ax.axis("off")
        ax.set_title(f"(y={y}, z={z}, g={g})")
        counts[g] += 1
        if all(c >= n_per_group for c in counts):
            break
    plt.show()



## 2) Backbone CNN (feature extractor + classification head)


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class SmallCNN(nn.Module):
    def __init__(self, feat_dim=128, num_classes=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        self.fc_feat = nn.Sequential(nn.Linear(64*7*7, feat_dim), nn.ReLU())
        self.fc_cls  = nn.Linear(feat_dim, num_classes)
    
    def forward(self, x, return_feat=False):
        h = self.conv(x)
        h = self.flatten(h)
        feat = self.fc_feat(h)
        logits = self.fc_cls(feat)
        if return_feat:
            return logits, feat
        return logits

model = SmallCNN().to(device)
model


SmallCNN(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc_feat): Sequential(
    (0): Linear(in_features=3136, out_features=128, bias=True)
    (1): ReLU()
  )
  (fc_cls): Linear(in_features=128, out_features=2, bias=True)
)

## 3) ERM Training


In [23]:

def train_erm(model, loaders, epochs=5, lr=1e-3, weight_decay=0.0):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": []}
    for ep in range(1, epochs+1):
        model.train()
        running = 0.0
        for x,y,_,_ in loaders["train"]:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()
            running += loss.item()*x.size(0)
        epoch_loss = running/len(loaders["train"].dataset)
        history["train_loss"].append(epoch_loss)
        print(f"Epoch {ep}/{epochs} - loss: {epoch_loss:.4f}")
    return model, history


## 4) Evaluation: Mean and Worst-Group Accuracy


In [24]:

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred, groups = [], [], []
    for x,y,_,g in loader:
        x = x.to(device)
        logits = model(x)
        preds = logits.argmax(1).cpu().numpy()
        y_true.extend(y.numpy())
        y_pred.extend(preds)
        groups.extend(g.numpy())
    y_true = np.array(y_true); y_pred=np.array(y_pred); groups=np.array(groups)
    
    mean_acc = (y_true==y_pred).mean()
    group_acc = {}
    for gi in np.unique(groups):
        mask = groups==gi
        group_acc[int(gi)] = (y_true[mask]==y_pred[mask]).mean()
    worst = min(group_acc.values())
    return mean_acc, worst, group_acc

def print_results(tag, mean_acc, worst, group_acc):
    print(f"[{tag}] mean_acc={mean_acc*100:.2f}%, worst_group={worst*100:.2f}%")
    for g,a in sorted(group_acc.items()):
        print(f"  - group {g}: {a*100:.2f}%")


## 5) End-to-End Training Run


In [26]:
# Run the full ERM training pipeline.
rho = 0.95
train_ds, val_clean, test_sp, test_clean, loaders = make_splits(rho=rho, batch_size=128)

model = SmallCNN().to(device)
model, hist = train_erm(model, loaders, epochs=3, lr=1e-3)

# Evaluate on all splits after training
for split in ["test_spurious", "test_clean", "val_clean"]:
    mean_acc, worst, group_acc = evaluate(model, loaders[split])
    print(f"{split}: mean_acc={mean_acc}, worst={worst}, group_acc={group_acc}")


Epoch 1/3 - loss: 0.0882
Epoch 2/3 - loss: 0.0295
Epoch 3/3 - loss: 0.0184
test_spurious: mean_acc=0.9928, worst=0.8601694915254238, group_acc={0: 0.9950270270270271, 1: 0.8601694915254238, 2: 0.9327731092436975, 3: 1.0}
test_clean: mean_acc=0.9608, worst=0.8809821057012068, group_acc={0: 0.9938974776240846, 1: 0.8809821057012068, 2: 0.9643835616438357, 3: 1.0}
val_clean: mean_acc=0.95375, worst=0.845, group_acc={0: 0.99, 1: 0.845, 2: 0.98, 3: 1.0}


## 6) Save Features for DFR (used in the next notebook)


In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    # Extract backbone embeddings (128-dim) for all samples in the loader.
    # Returns feature matrix X, labels y, and group assignments g.
    model.eval()
    Xs, ys, gs = [], [], []
    for x, y, _, g in loader:
        x = x.to(device)
        logits, feat = model(x, return_feat=True)
        Xs.append(feat.cpu())
        ys.append(y)
        gs.append(g)
    X = torch.cat(Xs).numpy()
    y = torch.cat(ys).numpy()
    g = torch.cat(gs).numpy()
    return X, y, g

out_dir = Path("./outputs"); out_dir.mkdir(exist_ok=True, parents=True)


